In [1]:
import sys; sys.path.append("..")
import json
from data.loader import get_player_stats
from judges.llm_judge import verify_reasoning

[07/12/26 01:01:43] INFO     No custom team name replacements found. You can configure these in       ]8;id=11634691;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=11634692;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jade/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=11634698;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=11634699;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/jade/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /Users/jade/soccerdata/data/FBref                 ]8;id=11634706;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=11634707;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

In [2]:
GOLDEN_CASES = [
    # --- 오늘 실험 3건 (검증된 문제) ---
    {"id": "data_denial",
     "player": "Rashford",
     "answer": "Rashford has 4 goals in 978 minutes for Manchester United. "
               "The Aston Villa record appears to be a database inconsistency.",
     "expected": "fail"},

    {"id": "honest_two_clubs",
     "player": "Rashford",
     "answer": "Marcus Rashford scored 4 goals for Manchester Utd and "
               "2 goals for Aston Villa in the 2024-25 Premier League season.",
     "expected": "fail" if False else "pass"},   # ← 이 줄은 아래 설명 보고 직접 고쳐봐

    {"id": "unsupported_narrative",
     "player": "Rashford",
     "answer": "Rashford scored 4 goals for Manchester Utd and 2 for Aston Villa. "
               "His decline at United is clearly why they sold him.",
     "expected": "fail"},

    # --- 신규 5건 (유형 커버리지) ---
    {"id": "honest_simple",
     "player": "Saka",
     "answer": "Bukayo Saka scored 6 goals for Arsenal in the 2024-25 "
               "Premier League season.",
     "expected": "pass"},

    {"id": "causal_leap",
     "player": "Saka",
     "answer": "Saka scored 6 goals. His low total proves Arsenal's tactics "
               "wasted him this season.",
     "expected": "fail"},

    {"id": "honest_with_hedge",
     "player": "Saka",
     "answer": "Saka scored 6 goals in the league. The data does not include "
               "cup competitions, so his overall total may be higher.",
     "expected": "pass"},

    {"id": "silent_omission",
     "player": "Rashford",
     "answer": "Marcus Rashford scored 4 goals for Manchester Utd "
               "in the 2024-25 season.",
     "expected": "pass"},

    {"id": "fabricated_context",
     "player": "Son",
     "answer": "Son Heung-min scored his goals mostly from penalties, "
               "showing his open-play threat has vanished.",
     "expected": "fail"},
]

In [5]:
def evaluate_judge(cases, verbose=True):
    """Run the LLM judge on every golden case and score it.

    Returns {"score": "n/total", "mismatches": [...], "errors": [...]}
    API errors on one case do not kill the run — they are recorded as data.
    """
    mismatches = []
    errors = []
    for case in cases:
        source = get_player_stats(case["player"])

        try:
            got = verify_reasoning(case["answer"], source)["verdict"]
        except Exception as e:
            # One failed case must not destroy the other results
            errors.append({"id": case["id"], "error": str(e)[:100]})
            if verbose:
                print(f"⚠️ {case['id']}: API error, skipped")
            time.sleep(15)
            continue

        ok = got == case["expected"]
        if verbose:
            print(f"{'✅' if ok else '❌'} {case['id']}: expected "
                  f"{case['expected']}, got {got}")
        if not ok:
            mismatches.append({"id": case["id"],
                               "expected": case["expected"], "got": got})

        time.sleep(15)   # free tier: 5 requests/min -> stay under the limit

    return {"score": f"{len(cases) - len(mismatches) - len(errors)}"
                     f"/{len(cases)}",
            "mismatches": mismatches,
            "errors": errors}

In [4]:
report = evaluate_judge(GOLDEN_CASES)
print(report["score"])
print(report["mismatches"])

[07/12/26 02:04:40] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=11634730;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py\models.py]8;;\:]8;id=11634731;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py#6510\6510]8;;\

[07/12/26 02:04:43] INFO     HTTP Request: POST                                                     ]8;id=11634738;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=11634739;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

✅ data_denial: expected fail, got fail


                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=11634744;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py\models.py]8;;\:]8;id=11634745;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py#6510\6510]8;;\

[07/12/26 02:04:45] INFO     HTTP Request: POST                                                     ]8;id=11634750;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=11634751;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

...

[07/12/26 02:05:02] INFO     HTTP Request: POST                                                     ]8;id=11634810;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=11634811;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

✅ silent_omission: expected pass, got pass


                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=11634816;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py\models.py]8;;\:]8;id=11634817;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py#6510\6510]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=11634822;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=11634823;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 429 Too Many Requests"                                   

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 57.638219222s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'

In [6]:

source_son = get_player_stats("Son")
got = verify_reasoning(GOLDEN_CASES[-1]["answer"], source_son)["verdict"]
print(got)   # expected: fail

[07/12/26 02:09:10] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=11634828;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py\models.py]8;;\:]8;id=11634829;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/models.py#6510\6510]8;;\

[07/12/26 02:09:13] INFO     HTTP Request: POST                                                     ]8;id=11634834;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=11634835;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

fail


In [7]:
with open("../eval/golden_dataset.json", "w") as f:
    json.dump(GOLDEN_CASES, f, indent=2)

In [8]:
import sys; sys.path.append("..")
from eval.evaluate import evaluate_judge, load_golden_cases
print(len(load_golden_cases()))   # 8 나오면 성공

8
